[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Stramasso/Projeto-Day-Trading-com-Python-Machine-Learning-2024-/blob/main/analise_rsi.ipynb)

# Análise de preços com RSI

Notebook com o passo a passo do script `rsi_analise.py`: baixa o histórico de um ativo da B3, calcula o Índice de Força Relativa (RSI) e marca os pregões em que o indicador sugere compra (RSI < 30), venda (RSI > 70) ou manutenção.

In [ ]:
%pip install -q yfinance

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf

ATIVO = 'CXSE3.SA'   # troque pelo ticker que quiser analisar
PERIODO_RSI = 14
SOBREVENDIDO, SOBRECOMPRADO = 30, 70

## 1. Histórico de preços
Preços de fechamento ajustados por proventos dos últimos dois anos.

In [ ]:
dados = yf.download(ATIVO, period='2y', auto_adjust=True, progress=False)
precos = dados['Close']
if isinstance(precos, pd.DataFrame):  # yfinance recente devolve colunas multinível
    precos = precos.iloc[:, 0]
precos = precos.rename('Preco').dropna()
precos.tail()

## 2. Cálculo do RSI
O RSI compara a média dos ganhos com a média das perdas em uma janela de 14 pregões:

$$RSI = 100 - \frac{100}{1 + \frac{\text{média dos ganhos}}{\text{média das perdas}}}$$

As médias usam a suavização de Wilder (média exponencial com $\alpha = 1/14$), que é a definição original do indicador.

In [ ]:
variacao = precos.diff()
ganhos = variacao.clip(lower=0)
perdas = -variacao.clip(upper=0)

media_ganhos = ganhos.ewm(alpha=1 / PERIODO_RSI, min_periods=PERIODO_RSI, adjust=False).mean()
media_perdas = perdas.ewm(alpha=1 / PERIODO_RSI, min_periods=PERIODO_RSI, adjust=False).mean()

rsi = 100 - 100 / (1 + media_ganhos / media_perdas)
rsi = rsi.where(media_perdas != 0, 100).where(media_ganhos.notna()).rename('RSI')

tabela = pd.concat([precos, rsi], axis=1).dropna()
tabela.tail()

## 3. Sinais

In [ ]:
def classificar_sinal(valor):
    if valor < SOBREVENDIDO:
        return 'Comprar'
    if valor > SOBRECOMPRADO:
        return 'Vender'
    return 'Manter'

tabela['Sinal'] = tabela['RSI'].apply(classificar_sinal)
tabela['Sinal'].value_counts()

## 4. Gráficos

In [ ]:
fig, (ax_preco, ax_rsi) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

ax_preco.plot(tabela.index, tabela['Preco'], color='#1f4e79', linewidth=1.2, label='Preço ajustado')
compras = tabela[tabela['Sinal'] == 'Comprar']
vendas = tabela[tabela['Sinal'] == 'Vender']
ax_preco.scatter(compras.index, compras['Preco'], marker='^', color='#2e7d32', label='Sinal de compra', zorder=3)
ax_preco.scatter(vendas.index, vendas['Preco'], marker='v', color='#c62828', label='Sinal de venda', zorder=3)
ax_preco.set_title(f'{ATIVO}: preço e sinais pelo RSI')
ax_preco.set_ylabel('Preço (R$)')
ax_preco.legend(loc='upper left')
ax_preco.grid(alpha=0.3)

ax_rsi.plot(tabela.index, tabela['RSI'], color='#6a1b9a', linewidth=1)
ax_rsi.axhline(SOBRECOMPRADO, linestyle='--', color='#c62828', alpha=0.6, label=f'Sobrecomprado ({SOBRECOMPRADO})')
ax_rsi.axhline(SOBREVENDIDO, linestyle='--', color='#2e7d32', alpha=0.6, label=f'Sobrevendido ({SOBREVENDIDO})')
ax_rsi.set_ylim(0, 100)
ax_rsi.set_ylabel('RSI')
ax_rsi.legend(loc='upper left')
ax_rsi.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 5. Leitura do último pregão

In [ ]:
ultimo = tabela.iloc[-1]
print(f"{tabela.index[-1]:%d/%m/%Y}: RSI {ultimo['RSI']:.1f} -> {ultimo['Sinal']}")

## Limitações

- O RSI isolado não é estratégia de investimento: em tendências fortes o ativo pode ficar sobrecomprado ou sobrevendido por muito tempo.
- Não há backtest, custos de corretagem nem slippage. Um próximo passo natural é medir o retorno de seguir os sinais contra simplesmente manter o ativo.